In [2]:
import os
import shutil
import pandas as pd
import numpy as np
import kagglehub

C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
raw_dir = "data/raw"
csv_path = os.path.join(raw_dir, "superstore.csv")
os.makedirs(raw_dir, exist_ok=True)

In [4]:
# download dataset
downloaded_path = kagglehub.dataset_download("vivek468/superstore-dataset-final")
print(f"Downloaded to: {downloaded_path}")

Downloaded to: C:\Users\DELL\.cache\kagglehub\datasets\vivek468\superstore-dataset-final\versions\1


In [5]:
files = os.listdir(downloaded_path)
csv_files = [f for f in files if f.endswith('.csv')]
if not csv_files:
    raise FileNotFoundError("No CSV files found in the downloaded Kaggle dataset.")
src_csv = os.path.join(downloaded_path, csv_files[0])

In [6]:
# copy csv file 0th entry to target path
shutil.copy(src_csv, csv_path)
print(f"Successfully copied dataset to {csv_path}")

Successfully copied dataset to data/raw\superstore.csv


In [7]:
df = pd.read_csv(csv_path, encoding='latin1')

In [8]:
# standardize column names to remove leading/trailing spaces 
df.rename(columns={c: c.strip() for c in df.columns}, inplace=True)

In [9]:
# dataset overview
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")

Shape: 9994 rows, 21 columns


In [10]:
df.head()
df.tail()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
9989,9990,CA-2014-110422,1/21/2014,1/23/2014,Second Class,TB-21400,Tom Boeckenhauer,Consumer,United States,Miami,...,33180,South,FUR-FU-10001889,Furniture,Furnishings,Ultra Door Pull Handle,25.248,3,0.2,4.1028
9990,9991,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,FUR-FU-10000747,Furniture,Furnishings,Tenex B1-RE Series Chair Mats for Low Pile Car...,91.960,2,0.0,15.6332
9991,9992,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,TEC-PH-10003645,Technology,Phones,Aastra 57i VoIP phone,258.576,2,0.2,19.3932
9992,9993,CA-2017-121258,2/26/2017,3/3/2017,Standard Class,DB-13060,Dave Brooks,Consumer,United States,Costa Mesa,...,92627,West,OFF-PA-10004041,Office Supplies,Paper,"It's Hot Message Books with Stickers, 2 3/4"" x 5""",29.600,4,0.0,13.3200
9993,9994,CA-2017-119914,5/4/2017,5/9/2017,Second Class,CC-12220,Chris Cortes,Consumer,United States,Westminster,...,92683,West,OFF-AP-10002684,Office Supplies,Appliances,"Acco 7-Outlet Masterpiece Power Center, Wihtou...",243.160,2,0.0,72.9480


In [16]:
# basic dataset info

print("\n=== Column Info & Null Values ===")
null_counts = df.isnull().sum()
types = df.dtypes
unique_counts = df.nunique()

profile_df = pd.DataFrame({
    'Data Type': types,
    'Null Count': null_counts,
    'Null %': (null_counts / len(df)) * 100,
    'Unique Count': unique_counts
})
print(profile_df.to_string())


=== Column Info & Null Values ===
              Data Type  Null Count  Null %  Unique Count
Row ID            int64           0     0.0          9994
Order ID            str           0     0.0          5009
Order Date          str           0     0.0          1237
Ship Date           str           0     0.0          1334
Ship Mode           str           0     0.0             4
Customer ID         str           0     0.0           793
Customer Name       str           0     0.0           793
Segment             str           0     0.0             3
Country             str           0     0.0             1
City                str           0     0.0           531
State               str           0     0.0            49
Postal Code       int64           0     0.0           631
Region              str           0     0.0             4
Product ID          str           0     0.0          1862
Category            str           0     0.0             3
Sub-Category        str           0  

In [17]:
unique_customers = df['Customer ID'].nunique()
unique_products = df['Order ID'].nunique()

print("Unique Customers:", unique_customers)
print("Unique Products:", unique_products)

Unique Customers: 793
Unique Products: 5009


In [18]:
# duplicate check

dup_count = df.duplicated().sum()
print(f"\nDuplicate Rows: {dup_count}")


Duplicate Rows: 0


In [19]:
# checking for anomalies if shipping date is earlier than order date
# finding if there are date columns to check
date_cols = [c for c in df.columns if 'date' in c.lower()]
print(f"Detected date columns: {date_cols}")
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

date_anomaly = 0
if len(date_cols) >= 2:
    
    # assuming order date and ship date are there
    
    order_col = [c for c in date_cols if 'order' in c.lower()][0]
    ship_col = [c for c in date_cols if 'ship' in c.lower()][0]
    date_anomaly = (df[ship_col] < df[order_col]).sum()
    print(f"Date anomalies ({ship_col} < {order_col}): {date_anomaly}")

Detected date columns: ['Order Date', 'Ship Date']
Date anomalies (Ship Date < Order Date): 0


In [20]:
# negative numeric columns check
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(f"Numeric columns: {list(numeric_cols)}")
for col in ['Sales', 'Quantity', 'Profit', 'Discount']:
    if col in df.columns:
        neg_count = (df[col] < 0).sum()
        print(f"Negative values in '{col}': {neg_count}")

Numeric columns: ['Row ID', 'Postal Code', 'Sales', 'Quantity', 'Discount', 'Profit']
Negative values in 'Sales': 0
Negative values in 'Quantity': 0
Negative values in 'Profit': 1871
Negative values in 'Discount': 0


In [22]:
postal_code_formats = (df['Postal Code'] < 10000).sum()

print(f"Postal codes with truncarted 0: {postal_code_formats}")

Postal codes with truncarted 0: 449


In [24]:
duplicate_combinations = (
    df.groupby(['Order ID', 'Product ID'])
      .size()
      .reset_index(name='count')
      .query('count > 1')
)

print(duplicate_combinations)

duplicate_count = df.duplicated(subset=['Order ID', 'Product ID']).sum()

print(f"Number of duplicate rows: {duplicate_count}")

            Order ID       Product ID  count
1751  CA-2015-103135  OFF-BI-10000069      2
4342  CA-2016-129714  OFF-PA-10001970      2
4560  CA-2016-137043  FUR-FU-10003664      2
4667  CA-2016-140571  OFF-PA-10001954      2
6295  CA-2017-118017  TEC-AC-10002006      2
7674  CA-2017-152912  OFF-ST-10003208      2
8520  US-2014-150119  FUR-CH-10002965      2
9105  US-2016-123750  TEC-AC-10004659      2
Number of duplicate rows: 8


In [25]:
has_duplicates = df.duplicated(subset=['Order ID', 'Product ID']).any()

print(has_duplicates)

True
